In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression,ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,root_mean_squared_error,confusion_matrix,roc_auc_score,log_loss
from sklearn.metrics import f1_score,accuracy_score,recall_score,precision_score,classification_report
from tqdm import tqdm
import os
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler,PolynomialFeatures
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
import datetime as dt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis,QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import BernoulliNB,GaussianNB
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,make_column_selector
from sklearn.tree import DecisionTreeClassifier,plot_tree,DecisionTreeRegressor
from sklearn.ensemble import BaggingClassifier,BaggingRegressor,VotingClassifier,VotingRegressor
from sklearn.metrics import accuracy_score

In [4]:
hr = pd.read_csv(r"../Datasets/HR_comma_sep.csv")
hr

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [7]:
X = hr.drop('left',axis=1)
y=hr['left']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26, stratify=hr['left'])

In [11]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")
trans = ColumnTransformer(
    transformers=[("OHE", ohe,make_column_selector(dtype_include=object))],remainder="passthrough",
    verbose_feature_names_out=False).set_output(transform="pandas")
X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)

In [12]:
knn = KNeighborsClassifier(n_neighbors=3)
bagg = BaggingClassifier(random_state=26,estimator=knn,n_estimators=15)
bagg.fit(X_trn_ohe,y_train)
y_pred = bagg.predict(X_tst_ohe)
accuracy_score(y_test,y_pred)

0.9437652811735942

In [14]:
n_neighbors = np.arange(1,20)
n_estimators = np.arange(1,20)
scores = []
for i in n_neighbors:
  knn = KNeighborsClassifier(n_neighbors=i)
  bagg = BaggingClassifier(random_state=26,estimator=knn,n_estimators=15)
  bagg.fit(X_trn_ohe,y_train)
  y_pred_prob = bagg.predict_proba(X_tst_ohe)
  scores.append([i,log_loss(y_test,y_pred_prob)])
df_score = pd.DataFrame(scores,columns = ['i','score'])
df_score.sort_values('score',ascending = False)

,i,score
0,1,0.795447
1,2,0.599117
2,3,0.500220
3,4,0.452442
4,5,0.412145
5,6,0.400136
6,7,0.395002
7,8,0.389878
8,9,0.378096
9,10,0.366318


In [16]:
dtc=DecisionTreeClassifier(max_depth = 4,random_state=26)
knn = KNeighborsClassifier(n_neighbors=3)
lr = LogisticRegression()
lda = LinearDiscriminantAnalysis()
nb = GaussianNB()


In [17]:
estims = [dtc,knn, lr, lda, nb]
scores = []

for e in tqdm(estims):
  for n in [10,15,50,100]:
    bagg = BaggingClassifier(random_state = 26, estimator =e ,n_estimators=n)
    bagg.fit(X_trn_ohe,y_train)
    y_pred_prob = bagg.predict_proba(X_tst_ohe)
    scores.append([e,n,log_loss(y_test,y_pred_prob)])
df_score = pd.DataFrame(scores,columns = ['e','n','log_loss'])
df_score.sort_values('log_loss',ascending = False)

  0%|                                                                                            | 0/5 [00:00<?, ?it/s]

 20%|████████████████▊                                                                   | 1/5 [00:01<00:07,  2.00s/it]

 40%|█████████████████████████████████▌                                                  | 2/5 [00:19<00:33, 11.24s/it]

 60%|██████████████████████████████████████████████████▍                                 | 3/5 [00:56<00:45, 22.84s/it]

 80%|███████████████████████████████████████████████████████████████████▏                | 4/5 [01:00<00:15, 15.65s/it]

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:02<00:00, 10.64s/it]

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:02<00:00, 12.55s/it]

,e,n,log_loss
16,GaussianNB(),10,0.723360
17,GaussianNB(),15,0.712025
18,GaussianNB(),50,0.710838
19,GaussianNB(),100,0.700531
4,KNeighborsClassifier(n_neighbors=3),10,0.524819
5,KNeighborsClassifier(n_neighbors=3),15,0.500220
6,KNeighborsClassifier(n_neighbors=3),50,0.450057
8,LogisticRegression(),10,0.430766
14,LinearDiscriminantAnalysis(),50,0.430667
15,LinearDiscriminantAnalysis(),100,0.430606
